# GNN-Pruning — Phase B: large-dataset **minibatch** re-run (Colab / A100)

Retrains the large node-classification datasets (Reddit, ogbn-arxiv, Yelp,
ogbn-products, Flickr) with **minibatch (neighbour-sampled) training**, which
fixes the full-batch under-training (1 epoch = 1 gradient step). The dense
baseline trains via the sampler; eval + Wanda activation-collection stay
full-batch on the sparse path, so the 4 pruning methods run unchanged.

The neighbour sampler is **pure PyTorch** — no `pyg-lib`/`torch-sparse` install.

**Before running:** `Runtime → A100 GPU` (Colab Pro), enable background execution.
The multi-seed *core* cells run separately on the Mac; this notebook does only the
five large datasets and hands back a results archive to merge locally.


## 1. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Set Runtime → A100 GPU"
print("GPU:", torch.cuda.get_device_name(0),
      "| VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1),
      "| torch:", torch.__version__)

## 2. Mount Drive (dataset cache only — results come back as a download)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DATA = '/content/drive/MyDrive/gnn-pruning/data'
os.makedirs(DRIVE_DATA, exist_ok=True)
print('dataset cache:', DRIVE_DATA)

## 3. Clone `main` + install (no pyg-lib needed)

In [ ]:
%cd /content
![ -d GNN-Pruning-Research ] || git clone --branch main https://github.com/Mike-Mans/GNN-Pruning-Research.git
%cd /content/GNN-Pruning-Research
!git fetch origin && git checkout main && git pull --ff-only

In [ ]:
# Cache datasets on Drive (download once); results stay in the repo dir.
import os, shutil
if not os.path.islink('data'):
    if os.path.exists('data'): shutil.rmtree('data')
    os.symlink(DRIVE_DATA, 'data')
!pip -q install torch_geometric ogb rdkit
!pip -q install -e .
print('install done')

## 4. Retrain large datasets (minibatch) — no-pruning first
Deletes the stale full-batch large cells so the idempotent runner retrains them with the minibatch path. `ogbn-products` now downloads headlessly. Expect a while on A100.

In [ ]:
import os, subprocess, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# stale full-batch large cells → remove so they retrain with minibatch
!rm -rf results/*/{reddit,ogbn-arxiv,yelp,ogbn-products,flickr}

CONFIGS = {
    'no-pruning':      'src/gnn_pruning/configs/no_pruning.yaml',
    'magnitude':       'src/gnn_pruning/configs/magnitude.yaml',
    'wanda-uniform':   'src/gnn_pruning/configs/wanda_uniform.yaml',
    'wanda-degree':    'src/gnn_pruning/configs/wanda_degree.yaml',
    'wanda-per-class': 'src/gnn_pruning/configs/wanda_per_class.yaml',
}
for method, cfg in CONFIGS.items():
    print(f'\n===== {method} =====', flush=True)
    rc = subprocess.run([sys.executable, '-m', 'gnn_pruning.cli', 'sweep',
                         '--method', method, '--config', cfg,
                         '--datasets', 'reddit,ogbn-arxiv,yelp,ogbn-products,flickr'],
                        env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f'{method} done (rc={rc})', flush=True)
print('\nPHASE B COMPLETE')

## 5. Did minibatch help? (dense baselines vs the old full-batch)

In [ ]:
import pandas as pd, glob, json
print('=== large-dataset dense baselines (minibatch-trained) ===')
for f in sorted(glob.glob('results/no-pruning/*/*/seed-0/split-0/metrics.json')):
    d = f.split('/')[2]
    if d in {'reddit','ogbn-arxiv','yelp','ogbn-products','flickr'}:
        m = json.load(open(f))
        print(f"  {d:14s}/{f.split('/')[3]:10s} {m['metric_value']:.3f}  (best epoch {m['epoch_of_best_val']})")
print('\n--- failures (expect only reddit/gat) ---')
!grep -h -A1 FAILED results/*/run.log | grep -i error | sort -u | head

## 6. Download the large-dataset results (merge into the local repo)

In [ ]:
import shutil, os, glob
LARGE_DS = ['reddit', 'ogbn-arxiv', 'yelp', 'ogbn-products', 'flickr']
os.makedirs('/content/large_export', exist_ok=True)
files = glob.glob('results/*/summary.csv') + glob.glob('results/*/run.log')
for d in LARGE_DS:                                  # Python glob has no brace expansion
    files += glob.glob(f'results/*/{d}/**/metrics.json', recursive=True)
for f in files:
    dst = '/content/large_export/' + f
    os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst)
shutil.make_archive('/content/gnn_large', 'zip', '/content/large_export')
from google.colab import files; files.download('/content/gnn_large.zip')

## Notes
- **Resuming after a disconnect:** re-run cells 1–4; finished cells are skipped.
- **Merge locally:** unzip into the repo, then rebuild summaries + the report:
  `python -m gnn_pruning.cli ...` is not needed — the summaries in the zip already
  reflect all cells on Drive; locally, regenerate `results_comprehensive.md`.
- reddit/gat stays infeasible (attention OOM); it's the only expected failure.
